# Multimodal Hateful Meme Detection — Colab Training

Train **frozen CLIP (ViT-B/32) + MLP** on the Facebook Hateful Memes dataset.

**Before you start:** Runtime → Change runtime type → **GPU**

**Dataset:** [parthplc/facebook-hateful-meme-dataset](https://www.kaggle.com/datasets/parthplc/facebook-hateful-meme-dataset) (via Kaggle)

In [ ]:
!git clone https://github.com/pjvk/Hateful-Meme-Detection-Using-CLIP.git
%cd Hateful-Meme-Detection-Using-CLIP

In [ ]:
!pip install -q -r requirements.txt
!pip install -q kagglehub

## Kaggle API token

1. Open [Kaggle Settings → API](https://www.kaggle.com/settings)
2. Click **Create New Token**
3. Paste the token in the cell below (do not commit a real token to GitHub)

In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "YOUR_KAGGLE_API_TOKEN_HERE"

## Download dataset from Kaggle → link as `data/`

In [ ]:
import kagglehub
from pathlib import Path
import os

path = kagglehub.dataset_download("parthplc/facebook-hateful-meme-dataset")
print("Path to dataset files:", path)

# Find folder with dev.jsonl + img/, then link:
root = next(Path(path).rglob("dev.jsonl")).parent
if Path("data").exists() or Path("data").is_symlink():
    Path("data").unlink() if Path("data").is_symlink() else None
Path("data").symlink_to(root.resolve())
!ls data

In [ ]:
!head -n 1 data/dev.jsonl
!python -m src.verify_data --data-dir data --split dev

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Optional: zero-shot CLIP baseline
!python -m src.zeroshot --data-dir data --split dev

In [ ]:
# Train CLIP + MLP (main model)
!python -m src.train --data-dir data --epochs 5 --batch-size 32 --lr 1e-3

In [ ]:
!python -m src.evaluate --data-dir data --split dev --checkpoint checkpoints/clip_mlp.pt

In [ ]:
from google.colab import files
files.download('checkpoints/clip_mlp.pt')